# Feature-wise Linear Modulation (FiLMConv) on Cora

Node Classification on Cora (Planetoid): Hypernetwork-driven feature-wise modulation of neighbor message passing. This notebook implements the approach with `FiLMConv` inside a `K3FiLMNet` model, trained with the Adam optimizer for 10 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `FiLMConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import PPI
from k3_node.loader import DataLoader

title = "Feature-wise Linear Modulation (FiLMConv) on PPI"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & Loaders
train_dataset = PPI(root="./data/PPI", split="train")
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

in_channels = train_dataset.num_features
out_channels = train_dataset.num_classes

# 2. FiLMConv Model Definition
class K3FiLMNet(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.FiLMConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.FiLMConv(hidden_channels, hidden_channels)
        self.conv3 = k3_layers.FiLMConv(hidden_channels, out_channels)

    def call(self, inputs, edge_index=None):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = ops.relu(self.conv1(x, edge_index))
        x = ops.relu(self.conv2(x, edge_index))
        return self.conv3(x, edge_index)

k3_model = K3FiLMNet(in_channels, 128, out_channels)

# 3. Model Compilation (Multi-label BCE)
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy(name="acc")],
)

# 4. Generator
def to_np(t, dtype=None):
    if hasattr(t, "cpu"):
        t = t.cpu()
    if hasattr(t, "detach"):
        t = t.detach()
    if hasattr(t, "numpy") and callable(t.numpy):
        t = t.numpy()
    return np.asarray(t, dtype=dtype)

def make_generator(loader):
    while True:
        for batch in loader:
            x = to_np(batch.x, dtype=np.float32)
            edge_index = to_np(batch.edge_index, dtype=np.int64)
            y = to_np(batch.y, dtype=np.float32)
            yield (x, edge_index), y

print(f"Training K3-Node FiLMConv model on {backend} backend...")
history = k3_model.fit(
    make_generator(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=10,
    verbose=1,
)

print("\n✓ K3-Node execution completed successfully!")